# 00 · Storage, sample data and the lesson map

Primary route: **S3 + EMR**. Alternate route: **Spark VM + HDFS**. Core notebooks use portable Spark DataFrames; notebook 11 adds AWS Glue Data Quality.

The sample orders use a fixed business date and arrive the following day. Keep the processing date aligned with the sample data so late and future-date checks remain reproducible.

The sequence is ingestion → schema → validation → business rules → reconciliation → gates → monitoring. A check measures a condition; a gate decides whether the pipeline may proceed.

Use a dedicated training prefix. This notebook reads the prepared sample files without changing them. Later outputs use unique run IDs. No bucket or cluster is created by these notebooks.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Read the prepared sample files
Upload the supplied `data/raw` folder before the session using `docs/DATA_SETUP.md`. No data generation is needed here. All inputs are read from `RAW_PATH`, which is `BASE_PATH` followed by `/raw`. The `orders` folder contains mixed valid and invalid records; `clean` contains a valid control order. Keep the folder names unchanged.


In [ ]:
print("Input location:", RAW_PATH)

# Inspect the mixed batch as text so malformed JSON remains visible.
mixed_orders = spark.read.text(f"{RAW_PATH}/orders")
print("Mixed-batch records:", mixed_orders.count())
mixed_orders.show(5, truncate=False)

# The clean batch contains a valid order for the successful pipeline run.
clean_orders = spark.read.json(f"{RAW_PATH}/clean")
clean_orders.show(truncate=False)


## Expected source ledger
There are 21 structurally valid dirty order rows (including one exact duplicate and one conflicting duplicate key), plus one malformed JSON line. O019 is a valid control record in the dirty batch. The separate clean fixture has one order. The invalid sample is deliberately much dirtier than any production tolerance.


In [ ]:
assert spark.read.text(f"{RAW_PATH}/orders").count() == 22
spark.read.text(f"{RAW_PATH}/csv_edge").show(truncate=False)
spark.read.text(f"{RAW_PATH}/json_edge").show(truncate=False)


## Checkpoint
Classify `quantity="two"`, unknown customer C999, and a missing shipment date. They occur at different layers. Predict which input modes preserve evidence before opening notebook 01. No AWS access keys belong in the notebook: use the EMR instance role or Glue execution role.
